# Combined ML + Strategy Predictions

Ensemble approach combining LSTM predictions and scalping strategy signals for improved trading decisions.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path.cwd().parent))

from src.data_collection.load_kaggle_data import load_kaggle_data
from src.preprocessing.clean_data import clean_ohlcv_data
from src.utils.data_split import split_data_by_date
from src.utils.config import DEFAULT_TICKERS, TRAIN_START, TRAIN_END, TEST_START, TEST_END

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import joblib

print("="*80)
print("COMBINED ML + STRATEGY BACKTESTING")
print("="*80)
print(f"Testing Period: {TEST_START} to {TEST_END}")
print("="*80)

COMBINED ML + STRATEGY BACKTESTING
Testing Period: 2024-01-01 to 2024-12-31


## Define Feature Engineering and Scalping Strategy

Replicate feature engineering and strategy logic from earlier notebooks.

In [2]:
# ======================================================
# SCALPING STRATEGY SIGNALS (Rule-Based)
# ======================================================
def add_scalping_signals(data):
    df = data.copy()

    # RSI
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    rsi = 100 - (100 / (1 + rs))

    # Moving averages
    sma_20 = df["Close"].rolling(20).mean()
    sma_50 = df["Close"].rolling(50).mean()

    # MACD
    ema_12 = df["Close"].ewm(span=12).mean()
    ema_26 = df["Close"].ewm(span=26).mean()
    macd = ema_12 - ema_26
    macd_signal = macd.ewm(span=9).mean()
    macd_hist = macd - macd_signal

    # BUY conditions
    buy_uptrend = (df["Close"] > sma_20) & (sma_20 > sma_50)
    buy_rsi = rsi < 40
    buy_macd = (macd > 0) & (macd_hist > 0)

    close_20_high = df["Close"].rolling(20).max()
    buy_strength = df["Close"] > 0.95 * close_20_high

    buy_signal = (
        (buy_uptrend & buy_rsi) |
        (buy_uptrend & buy_macd) |
        (buy_uptrend & buy_strength)
    )

    # SELL conditions
    sell_downtrend = (df["Close"] < sma_20) | (sma_20 < sma_50)
    sell_rsi = rsi > 60
    sell_macd = (macd < 0) & (macd_hist < 0)

    sell_signal = (
        (sell_downtrend & sell_rsi) |
        (sell_downtrend & sell_macd)
    )

    # Final signal
    signal = pd.Series(0, index=df.index)
    signal[buy_signal] = 1
    signal[sell_signal] = -1
    signal[(buy_signal) & (sell_signal)] = 1

    df["strategy_signal"] = signal
    return df


# ======================================================
# FEATURE ENGINEERING (ML + Trading Aligned)
# ======================================================
def add_basic_features(data, horizon=3, cost=0.0003):
    df = data.copy()

    # Returns
    df["returns"] = df["Close"].pct_change()
    df["log_returns"] = np.log(df["Close"] / df["Close"].shift(1))

    # Trend
    sma_10 = df["Close"].rolling(10).mean()
    sma_20 = df["Close"].rolling(20).mean()

    df["trend_10"] = (df["Close"] - sma_10) / sma_10
    df["trend_20"] = (df["Close"] - sma_20) / sma_20
    df["trend_diff"] = (sma_10 - sma_20) / sma_20

    # Price action
    df["range_pct"] = (df["High"] - df["Low"]) / df["Close"]
    df["body_pct"] = (df["Close"] - df["Open"]) / df["Close"]
    df["body_abs"] = df["body_pct"].abs()

    # Volatility regime
    df["volatility_10"] = df["returns"].rolling(10).std()
    df["vol_ratio"] = df["volatility_10"] / df["volatility_10"].rolling(50).mean()
    df["high_vol"] = (df["vol_ratio"] > 1.0).astype(int)

    # RSI (0–1)
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    df["RSI"] = (100 - (100 / (1 + rs))) / 100.0

    # Volume
    if "Volume" in df.columns and df["Volume"].sum() > 0:
        vol_sma = df["Volume"].rolling(20).mean()
        df["Volume_norm"] = np.log1p(df["Volume"] / (vol_sma + 1e-8))
    else:
        df["Volume_norm"] = 0.0

    # Target: forward return beyond cost
    future_return = (df["Close"].shift(-horizon) - df["Close"]) / df["Close"]
    df["target"] = (future_return > cost).astype(int)

    df.dropna(inplace=True)
    return df






## Load Data and Generate Both ML + Strategy Predictions

For first ticker, compare:
1. ML predictions (LSTM only)
2. Strategy signals (technical rules only)
3. Combined predictions (voting ensemble)

In [3]:
ticker = DEFAULT_TICKERS[0]
print(f"\n{'='*80}")
print(f"ANALYZING {ticker}")
print(f"{'='*80}")

# Load and prepare data
raw_data = load_kaggle_data(ticker)
cleaned_data = clean_ohlcv_data(raw_data)
train_data, test_data = split_data_by_date(cleaned_data)

print(f"Train data: {train_data.shape}")
print(f"Test data: {test_data.shape}")

# Feature engineering for ML
# ------------------------------------------------------
# Apply strategy FIRST (raw data)
# ------------------------------------------------------
train_with_signals = add_scalping_signals(train_data)
test_with_signals  = add_scalping_signals(test_data)

# ------------------------------------------------------
# Then apply feature engineering (keeps alignment)
# ------------------------------------------------------
train_with_features = add_basic_features(train_with_signals)
test_with_features  = add_basic_features(test_with_signals)

print(f"Train with features: {train_with_features.shape}")
print(f"Test with features:  {test_with_features.shape}")



ANALYZING NIFTY BANK
2025-12-26 17:45:10 - src.data_collection.load_kaggle_data - INFO - Loading NIFTY BANK from C:\Users\Sunay Bhattacharjee\Desktop\AlgoTrading bot project\SnowMore\algo-trading-project\data\raw\NIFTY BANK_minute.csv
2025-12-26 17:45:11 - src.data_collection.load_kaggle_data - INFO - Loaded 975275 rows for NIFTY BANK from 2015-01-09 09:15:00 to 2025-07-25 15:29:00
2025-12-26 17:45:11 - src.preprocessing.clean_data - INFO - Volume column largely zero — skipping volume filter
2025-12-26 17:45:11 - src.preprocessing.clean_data - INFO - Removed 19506 outliers from Open
2025-12-26 17:45:11 - src.preprocessing.clean_data - INFO - Removed 19114 outliers from High
2025-12-26 17:45:11 - src.preprocessing.clean_data - INFO - Removed 18733 outliers from Low
2025-12-26 17:45:11 - src.preprocessing.clean_data - INFO - Removed 18357 outliers from Close
2025-12-26 17:45:11 - src.preprocessing.clean_data - INFO - Cleaned OHLCV data → 899565 rows | 2015-01-09 09:15:00 to 2025-04-16 0

In [4]:
# ======================================================
# STEP 1: ML PREDICTIONS (XGBoost - IMPROVED)
# ======================================================
print("\n" + "="*80)
print("STEP 1: XGBoost ML MODEL PREDICTIONS (IMPROVED)")
print("="*80)

# ------------------------------------------------------
# Feature selection
# ------------------------------------------------------
feature_cols = [
    col for col in train_with_features.columns
    if col not in ['target', 'Open', 'High', 'Low', 'Close', 'Volume']
]

X_train = train_with_features[feature_cols]
y_train = train_with_features['target']

X_test = test_with_features[feature_cols]
y_test = test_with_features['target']

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

# ------------------------------------------------------
# Time-based validation split (NO leakage)
# ------------------------------------------------------
val_size = int(0.8 * len(X_train))
X_tr, X_val = X_train.iloc[:val_size], X_train.iloc[val_size:]
y_tr, y_val = y_train.iloc[:val_size], y_train.iloc[val_size:]

# ------------------------------------------------------
# XGBoost model (tuned for noisy financial data)
# ------------------------------------------------------
from xgboost import XGBClassifier
from xgboost.callback import EarlyStopping

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=20,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    verbosity=0
)

print("\nTraining XGBoost model...")

xgb_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)



# ------------------------------------------------------
# Probabilities
# ------------------------------------------------------
y_val_prob = xgb_model.predict_proba(X_val)[:, 1]
y_test_prob = xgb_model.predict_proba(X_test)[:, 1]

# ------------------------------------------------------
# Threshold optimization (EDGE-BASED, validation only)
# ------------------------------------------------------
best_threshold = 0.5
best_score = -np.inf

baseline = y_val.mean()   # base probability of positive target

for t in np.arange(0.35, 0.65, 0.02):
    preds = (y_val_prob > t).astype(int)
    trade_rate = preds.mean()

    # avoid extreme over/under trading
    if trade_rate < 0.05 or trade_rate > 0.4:
        continue

    if preds.sum() > 0:
        win_rate = y_val[preds == 1].mean()
    else:
        win_rate = 0.0

    edge = win_rate - baseline

    # penalize tiny sample sizes
    score = edge * np.sqrt(preds.sum())

    if score > best_score:
        best_score = score
        best_threshold = t


# ------------------------------------------------------
# Final test predictions
# ------------------------------------------------------
y_test_pred = (y_test_prob > best_threshold).astype(int)

ml_accuracy = accuracy_score(y_test, y_test_pred)
ml_auc = roc_auc_score(y_test, y_test_prob)
ml_f1 = f1_score(y_test, y_test_pred, zero_division=0)

print(f"\n✓ ML Threshold: {best_threshold:.2f}")
print(f"✓ ML Test Accuracy: {ml_accuracy:.4f}")
print(f"✓ ML Test AUC:      {ml_auc:.4f}")
print(f"✓ ML Test F1:       {ml_f1:.4f}")



STEP 1: XGBoost ML MODEL PREDICTIONS (IMPROVED)
X_train: (791543, 14)
X_test:  (80848, 14)
y_train: (791543,)
y_test:  (80848,)

Training XGBoost model...

✓ ML Threshold: 0.35
✓ ML Test Accuracy: 0.6588
✓ ML Test AUC:      0.6041
✓ ML Test F1:       0.3402


In [5]:
test_eval = test_with_features.copy()

test_eval["ml_prob"] = y_test_prob[:len(test_eval)]
test_eval["ml_entry"] = (test_eval["ml_prob"] > best_threshold).astype(int)

test_eval["strategy_entry"] = (test_eval["strategy_signal"] == 1).astype(int)

test_eval["combined_entry"] = (
    (test_eval["strategy_entry"] == 1) &
    (test_eval["ml_entry"] == 1)
).astype(int)

baseline = y_test[:len(test_eval)].mean()

def evaluate(name, col):
    entries = test_eval[col]
    trade_rate = entries.mean()
    win_rate = y_test[:len(test_eval)][entries == 1].mean() if entries.sum() > 0 else 0
    edge = win_rate - baseline
    print(f"\n--- {name} ---")
    print(f"Trade rate: {trade_rate:.2%}")
    print(f"Win rate:   {win_rate:.2%}")
    print(f"Edge:       {edge:.2%}")

evaluate("STRATEGY ONLY", "strategy_entry")
evaluate("ML ONLY", "ml_entry")
evaluate("STRATEGY + ML", "combined_entry")



--- STRATEGY ONLY ---
Trade rate: 30.47%
Win rate:   26.42%
Edge:       -1.74%

--- ML ONLY ---
Trade rate: 23.55%
Win rate:   37.35%
Edge:       9.19%

--- STRATEGY + ML ---
Trade rate: 4.92%
Win rate:   36.13%
Edge:       7.97%


In [6]:
# ======================================================
# STEP 2: STRATEGY AS CONTEXT FILTER (ML-ALIGNED)
# ======================================================
print("\n" + "="*80)
print("STEP 2: STRATEGY AS CONTEXT FILTER (ML-ALIGNED)")
print("="*80)

# Base evaluation frame (must already be aligned)
test_eval = test_with_features.copy()

# ------------------------------------------------------
# Entries
# ------------------------------------------------------
test_eval["strategy_entry"] = (test_eval["strategy_signal"] == 1).astype(int)
test_eval["ml_entry"] = (y_test_prob > best_threshold).astype(int)

test_eval["combined_entry"] = (
    (test_eval["strategy_entry"] == 1) &
    (test_eval["ml_entry"] == 1)
).astype(int)

# Align target
y_target = y_test.values[:len(test_eval)]
baseline = y_target.mean()

# ------------------------------------------------------
# Evaluation helper
# ------------------------------------------------------
def evaluate(name, entry_col):
    entries = test_eval[entry_col]
    trade_rate = entries.mean()
    
    if entries.sum() > 0:
        win_rate = y_target[entries == 1].mean()
    else:
        win_rate = 0.0

    edge = win_rate - baseline

    print(f"\n--- {name} ---")
    print(f"Trade rate: {trade_rate:.2%}")
    print(f"Win rate:   {win_rate:.2%}")
    print(f"Edge:       {edge:.2%}")

# ------------------------------------------------------
# Results
# ------------------------------------------------------
evaluate("STRATEGY ONLY", "strategy_entry")
evaluate("ML ONLY", "ml_entry")
evaluate("STRATEGY + ML", "combined_entry")



STEP 2: STRATEGY AS CONTEXT FILTER (ML-ALIGNED)

--- STRATEGY ONLY ---
Trade rate: 30.47%
Win rate:   26.42%
Edge:       -1.74%

--- ML ONLY ---
Trade rate: 23.55%
Win rate:   37.35%
Edge:       9.19%

--- STRATEGY + ML ---
Trade rate: 4.92%
Win rate:   36.13%
Edge:       7.97%


## Key Findings

Summary of the combined ML + Strategy approach compared to individual techniques.

In [7]:
# =====================================================
# BACKTEST CONFIG (MAX-RETURN READY)
# =====================================================

INITIAL_CAPITAL = 1_000_000
RISK_FREE_RATE = 0.0

HORIZON =60

# -------------------------------
# ML ENTRY (TAIL ONLY)
# -------------------------------
ENTRY_Q = 0.96    # trade only top 5% signals

# -------------------------------
# EXECUTION CONTROLS
# -------------------------------
STOP_LOSS = 0.004
COST_PER_TRADE = 0.00015

# Position sizing (convex)
SIZE_EXPONENT = 3
MAX_POSITION = 1.0



# Trade management
COOLDOWN = HORIZON//2


In [8]:
ticker = "NIFTY BANK"
print(f"\nBacktesting: {ticker}")

# --------------------------------------------------
# Load & clean
# --------------------------------------------------
raw = load_kaggle_data(ticker)
cleaned = clean_ohlcv_data(raw)
train_data, test_data = split_data_by_date(cleaned)

# --------------------------------------------------
# CONTINUOUS FEATURE PIPELINE (CRITICAL FIX)
# --------------------------------------------------
# Combine train + test to preserve rolling context
full_data = pd.concat([train_data, test_data], axis=0)

# Apply features on full history
full_with_signals = add_scalping_signals(full_data)
full_features = add_basic_features(full_with_signals)

# Slice back test portion ONLY
test_df = full_features.loc[test_data.index]

# --------------------------------------------------
# ML inputs
# --------------------------------------------------
feature_cols = [
    c for c in test_df.columns
    if c not in ["target", "Open", "High", "Low", "Close", "Volume"]
]

X_test = test_df[feature_cols]
y_test = test_df["target"]
prices = test_df["Close"].values



Backtesting: NIFTY BANK
2025-12-26 17:45:35 - src.data_collection.load_kaggle_data - INFO - Loading NIFTY BANK from C:\Users\Sunay Bhattacharjee\Desktop\AlgoTrading bot project\SnowMore\algo-trading-project\data\raw\NIFTY BANK_minute.csv
2025-12-26 17:45:36 - src.data_collection.load_kaggle_data - INFO - Loaded 975275 rows for NIFTY BANK from 2015-01-09 09:15:00 to 2025-07-25 15:29:00
2025-12-26 17:45:36 - src.preprocessing.clean_data - INFO - Volume column largely zero — skipping volume filter
2025-12-26 17:45:36 - src.preprocessing.clean_data - INFO - Removed 19506 outliers from Open
2025-12-26 17:45:36 - src.preprocessing.clean_data - INFO - Removed 19114 outliers from High
2025-12-26 17:45:36 - src.preprocessing.clean_data - INFO - Removed 18733 outliers from Low
2025-12-26 17:45:36 - src.preprocessing.clean_data - INFO - Removed 18357 outliers from Close
2025-12-26 17:45:36 - src.preprocessing.clean_data - INFO - Cleaned OHLCV data → 899565 rows | 2015-01-09 09:15:00 to 2025-04-1

In [9]:
from sklearn.preprocessing import StandardScaler

# --------------------------------------------------
# TRAIN FEATURES (DEFINE FEATURE SPACE HERE)
# --------------------------------------------------
train_with_signals = add_scalping_signals(train_data)
train_df = add_basic_features(train_with_signals)

feature_cols = [
    c for c in train_df.columns
    if c not in ["target", "Open", "High", "Low", "Close", "Volume"]
]

X_train = train_df[feature_cols]
y_train = train_df["target"]

# --------------------------------------------------
# SCALE (FIT ON TRAIN ONLY)
# --------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --------------------------------------------------
# MODEL
# --------------------------------------------------
model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=20,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    verbosity=0
)

model.fit(X_train_scaled, y_train)

# --------------------------------------------------
# ML PROBABILITIES
# --------------------------------------------------
ml_prob = model.predict_proba(X_test_scaled)[:, 1]
print("Test AUC:", roc_auc_score(y_test, ml_prob))


Test AUC: 0.6049132125626437


In [10]:
def position_size(prob, threshold):
    """
    Convert ML probability into position size [0, 1]
    """
    size = (prob - threshold) / (1 - threshold)
    return np.clip(size, 0, 1)


In [11]:

capital = INITIAL_CAPITAL
equity_curve = []
trades = []

# ---------------------------------
# ENTRY THRESHOLD (TAIL ONLY)
# ---------------------------------
ENTRY_THRESHOLD = np.quantile(ml_prob, ENTRY_Q)

i = 0
n = len(prices)

while i < n - HORIZON:

    prob = ml_prob[i]

    # --------------------------------------------------
    # 1. SKIP LOW-CONFIDENCE SIGNALS
    # --------------------------------------------------
    if prob <= ENTRY_THRESHOLD:
        equity_curve.append(capital)
        i += 1
        continue

    # --------------------------------------------------
    # 2. VOLATILITY-BASED RISK (KNOWN AT ENTRY)
    # --------------------------------------------------
    vol = test_df["volatility_10"].iloc[i]

    STOP_LOSS = 1.2 * vol        # adaptive risk
    MAX_HOLD  = HORIZON          # hard time cap

    # --------------------------------------------------
    # 3. POSITION SIZING
    # --------------------------------------------------
    edge_strength = (prob - ENTRY_THRESHOLD) / (1 - ENTRY_THRESHOLD)
    size = np.clip(edge_strength ** SIZE_EXPONENT, 0.25, 1.0)

    risk_adj = np.clip(0.01 / vol, 0.5, 2.0)

    position_value = capital * size * risk_adj
    entry_price = prices[i]

    # --------------------------------------------------
    # 4. EXIT LOGIC (STOP LOSS + TIME ONLY)
    # --------------------------------------------------
    exit_price = prices[i + MAX_HOLD]
    exit_idx   = i + MAX_HOLD

    for j in range(1, MAX_HOLD + 1):
        price = prices[i + j]

        # STOP LOSS ONLY
        if price <= entry_price * (1 - STOP_LOSS):
            exit_price = entry_price * (1 - STOP_LOSS)
            exit_idx = i + j
            break

    # --------------------------------------------------
    # 5. PnL CALCULATION (ONCE)
    # --------------------------------------------------
    ret = (exit_price - entry_price) / entry_price
    net_ret = ret - COST_PER_TRADE

    pnl = position_value * net_ret
    capital += pnl

    trades.append({
        "entry_idx": i,
        "exit_idx": exit_idx,
        "prob": prob,
        "size": size,
        "return": net_ret,
        "pnl": pnl,
        "capital": capital
    })

    equity_curve.append(capital)

    # --------------------------------------------------
    # 6. COOLDOWN
    # --------------------------------------------------
    i += COOLDOWN


In [12]:
equity = pd.Series(equity_curve)
returns = equity.pct_change().dropna()

total_return = (equity.iloc[-1] / equity.iloc[0]) - 1
max_dd = ((equity / equity.cummax()) - 1).min()

sharpe = (
    returns.mean() / returns.std()
    if returns.std() > 0 else 0
) * np.sqrt(252 * 6.5 * 60)   # intraday annualization

profit_factor = (
    sum(t["pnl"] for t in trades if t["pnl"] > 0) /
    abs(sum(t["pnl"] for t in trades if t["pnl"] < 0))
    if any(t["pnl"] < 0 for t in trades) else np.inf
)

print("\n================ BACKTEST SUMMARY ================")
print(f"Final Capital:     ₹{capital:,.0f}")
print(f"Total Return:      {total_return*100:.2f}%")
print(f"Max Drawdown:      {max_dd*100:.2f}%")
print(f"Sharpe Ratio:      {sharpe:.2f}")
print(f"Profit Factor:    {profit_factor:.2f}")
print(f"Total Trades:     {len(trades)}")



================ BACKTEST SUMMARY ================
Final Capital:     ₹1,176,320
Total Return:      17.53%
Max Drawdown:      -1.37%
Sharpe Ratio:      4.80
Profit Factor:    1.77
Total Trades:     558


In [13]:
trade_returns = [t["return"] for t in trades]

print("\nTrade Stats")
print(f"Win rate: {np.mean(np.array(trade_returns) > 0)*100:.2f}%")
print(f"Avg win:  {np.mean([r for r in trade_returns if r > 0])*100:.2f}%")
print(f"Avg loss: {np.mean([r for r in trade_returns if r < 0])*100:.2f}%")



Trade Stats
Win rate: 30.11%
Avg win:  0.45%
Avg loss: -0.11%


In [14]:
import numpy as np
import pandas as pd

# ======================================================
# SCALPING FEATURES (STRATEGY + ML, LIVE-SAFE)
# ======================================================
def add_scalping_features(data, horizon=3, cost=0.0003, make_target=False):
    df = data.copy()

    # --------------------------------------------------
    # RETURNS
    # --------------------------------------------------
    df["returns"] = df["Close"].pct_change()
    df["log_returns"] = np.log(df["Close"] / df["Close"].shift(1))

    # --------------------------------------------------
    # TREND
    # --------------------------------------------------
    sma_10 = df["Close"].rolling(10).mean()
    sma_20 = df["Close"].rolling(20).mean()
    sma_50 = df["Close"].rolling(50).mean()

    df["trend_10"] = (df["Close"] - sma_10) / sma_10
    df["trend_20"] = (df["Close"] - sma_20) / sma_20
    df["trend_diff"] = (sma_10 - sma_20) / sma_20

    # --------------------------------------------------
    # PRICE ACTION
    # --------------------------------------------------
    df["range_pct"] = (df["High"] - df["Low"]) / df["Close"]
    df["body_pct"] = (df["Close"] - df["Open"]) / df["Close"]
    df["body_abs"] = df["body_pct"].abs()

    # --------------------------------------------------
    # VOLATILITY REGIME
    # --------------------------------------------------
    df["volatility_10"] = df["returns"].rolling(10).std()
    df["vol_ratio"] = df["volatility_10"] / df["volatility_10"].rolling(50).mean()
    df["high_vol"] = (df["vol_ratio"] > 1.0).astype(int)

    # --------------------------------------------------
    # RSI (0–1 scaled)
    # --------------------------------------------------
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    df["RSI"] = (100 - (100 / (1 + rs))) / 100.0

    # --------------------------------------------------
    # VOLUME
    # --------------------------------------------------
    if "Volume" in df.columns and df["Volume"].sum() > 0:
        vol_sma = df["Volume"].rolling(20).mean()
        df["Volume_norm"] = np.log1p(df["Volume"] / (vol_sma + 1e-8))
    else:
        df["Volume_norm"] = 0.0

    # --------------------------------------------------
    # RULE-BASED STRATEGY SIGNAL
    # --------------------------------------------------
    ema_12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema_26 = df["Close"].ewm(span=26, adjust=False).mean()
    macd = ema_12 - ema_26
    macd_signal = macd.ewm(span=9, adjust=False).mean()
    macd_hist = macd - macd_signal

    buy_uptrend = (df["Close"] > sma_20) & (sma_20 > sma_50)
    buy_rsi = df["RSI"] < 0.40
    buy_macd = (macd > 0) & (macd_hist > 0)

    close_20_high = df["Close"].rolling(20).max()
    buy_strength = df["Close"] > 0.95 * close_20_high

    sell_downtrend = (df["Close"] < sma_20) | (sma_20 < sma_50)
    sell_rsi = df["RSI"] > 0.60
    sell_macd = (macd < 0) & (macd_hist < 0)

    signal = pd.Series(0, index=df.index)
    signal[(buy_uptrend & buy_rsi) | (buy_uptrend & buy_macd) | (buy_uptrend & buy_strength)] = 1
    signal[(sell_downtrend & sell_rsi) | (sell_downtrend & sell_macd)] = -1

    df["strategy_signal"] = signal

    # --------------------------------------------------
    # TARGET (BACKTEST ONLY)
    # --------------------------------------------------
    if make_target:
        future_return = (df["Close"].shift(-horizon) - df["Close"]) / df["Close"]
        df["target"] = (future_return > cost).astype(int)

    return df

In [15]:
def fetch_intraday_data(period="1d"):
    df = yf.download(
        SYMBOL,
        period=period,
        interval="1m",
        progress=False
    )

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.columns = df.columns.astype(str).str.strip().str.capitalize()

    if "Adj close" in df.columns and "Close" not in df.columns:
        df.rename(columns={"Adj close": "Close"}, inplace=True)

    return df.dropna()


In [16]:
import yfinance as yf
SYMBOL = "^NSEBANK"


df_raw = fetch_intraday_data(period="8d")  # today (use "5d" if you want more)

df_feat = add_basic_features(
    df_raw,
    horizon=HORIZON,
    cost=COST_PER_TRADE
)

df_sig = add_scalping_signals(df_raw)
df_feat["strategy_signal"] = df_sig["strategy_signal"]

# Safety check
assert set(feature_cols).issubset(df_feat.columns)


In [17]:
paper_capital = INITIAL_CAPITAL
position = 0

entry_price = None
entry_time = None
qty = 0.0
invested_amount = 0.0
ENTRY_THRESHOLD=0.22
HORIZON=12
TAKE_PROFIT=0.0045
STOP_LOSS=0.003

INVEST_FRACTION = 0.50 # 50% capital per trade 
MIN_POSITION_FRACTION = 0.02 # safety filter

trades = []

for i in range(len(df_feat)):

    # warm-up guard (same as live)
    if i < 30:
        continue

    row = df_feat.iloc[i]
    ts = row.name
    current_price = row["Close"]

    # ------------------------------
    # ML PREDICTION (exact same)
    # ------------------------------
    X_live = scaler.transform(df_feat.iloc[[i]][feature_cols])
    prob = model.predict_proba(X_live)[0, 1]

    # ------------------------------
    # ENTRY (same logic)
    # ------------------------------
    if position == 0 and prob >= ENTRY_THRESHOLD:
        position_fraction = (prob ** SIZE_EXPONENT) * MAX_POSITION
        MIN_POSITION_FRACTION = 0.02  # 2%

        if position_fraction < MIN_POSITION_FRACTION:
            continue

        invested_amount = paper_capital * 0.5
        qty = invested_amount / current_price

        entry_price = current_price
        entry_time = ts
        position = 1

        trades.append({
            "type": "BUY",
            "time": ts,
            "price": current_price,
            "prob": prob
        })

    # ------------------------------
    # EXIT (same logic)
    # ------------------------------
    elif position == 1: 
        pnl_pct = (current_price - entry_price) / entry_price 
        hold_minutes = int((ts - entry_time).total_seconds() // 60) 
        exit_reason = ( pnl_pct <= -STOP_LOSS or pnl_pct >= TAKE_PROFIT or hold_minutes >= HORIZON ) 
        if exit_reason: 
            pnl_cash = (current_price - entry_price) * qty 
            paper_capital += pnl_cash - invested_amount * COST_PER_TRADE
            trades.append({ "type": "SELL", "time": ts, "price": current_price, "pnl": pnl_cash }) 
            position = 0 
            entry_price = None
            entry_time = None 
            qty = 0.0 
            invested_amount = 0.0

In [18]:
trades_df = pd.DataFrame(trades)

sell_trades = trades_df[trades_df["type"] == "SELL"].copy()

print("Number of SELL trades:", len(sell_trades))


Number of SELL trades: 48


In [19]:
total_trades = len(sell_trades)

win_trades = sell_trades[sell_trades["pnl"] > 0]
loss_trades = sell_trades[sell_trades["pnl"] <= 0]

win_rate = len(win_trades) / total_trades if total_trades > 0 else 0.0

avg_win = win_trades["pnl"].mean() if len(win_trades) > 0 else 0.0
avg_loss = loss_trades["pnl"].mean() if len(loss_trades) > 0 else 0.0

print(f"Total trades      : {total_trades}")
print(f"Win rate          : {win_rate*100:.2f}%")
print(f"Avg win (₹)       : {avg_win:.2f}")
print(f"Avg loss (₹)      : {avg_loss:.2f}")
pct_return = (paper_capital - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100
print(f"Return (%): {pct_return:.6f}%")



final_capital = paper_capital

print(f"Initial Capital : {INITIAL_CAPITAL:,.2f}")
print(f"Final Capital   : {final_capital:,.2f}")
print(f"Net PnL         : {final_capital - INITIAL_CAPITAL:,.2f}")


Total trades      : 48
Win rate          : 66.67%
Avg win (₹)       : 348.61
Avg loss (₹)      : -176.94
Return (%): 0.471441%
Initial Capital : 1,000,000.00
Final Capital   : 1,004,714.41
Net PnL         : 4,714.41


In [20]:
import yfinance as yf
import pandas as pd
import pytz
from datetime import datetime

IST = pytz.timezone("Asia/Kolkata")
SYMBOL = "^NSEBANK"

def fetch_today_data():
    df = yf.download(
        SYMBOL,
        period="1d",
        interval="1m",
        progress=False
    )

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.columns = df.columns.str.capitalize()

    # ---- timezone-safe handling
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")

    df.index = df.index.tz_convert(IST)

    # ---- FORCE only today's IST data
    today_ist = datetime.now(IST).date()
    df = df[df.index.date == today_ist]

    return df.dropna()
df_today = fetch_today_data()

num_days = df_today.index.normalize().nunique()
print("Number of unique days:", num_days)
df_today_feat = add_basic_features(
    df_today,
    horizon=HORIZON,
    cost=COST_PER_TRADE
)

# Optional if you use rule-based signals
df_today_feat = add_scalping_signals(df_today_feat)




Number of unique days: 1


In [21]:

# ===============================
# OFFLINE PAPER TRADING (REPLAY)
# ===============================

paper_capital = INITIAL_CAPITAL
position = 0

entry_price = None
entry_time = None
qty = 0.0
invested_amount = 0.0

ENTRY_THRESHOLD = 0.22
HORIZON = 12
TAKE_PROFIT = 0.0045
STOP_LOSS = 0.003

INVEST_FRACTION = 0.50        # 50% capital per trade
MIN_POSITION_FRACTION = 0.02  # safety filter

trades = []

# ---- IMPORTANT: replay ONLY on today's features
df_replay = df_today_feat.copy()

# ---- Sanity check (never skip this)
assert df_replay.index.normalize().nunique() == 1, "Replay is NOT 1-day data!"

for i in range(len(df_replay)):

    # indicator warm-up (same as live)
    if i < 30:
        continue

    row = df_replay.iloc[i]
    ts = row.name
    current_price = row["Close"]

    # ==============================
    # ML PREDICTION (same as live)
    # ==============================
    X_live = scaler.transform(df_replay.iloc[[i]][feature_cols])
    prob = model.predict_proba(X_live)[0, 1]

    # ==============================
    # ENTRY
    # ==============================
    if position == 0 and prob >= ENTRY_THRESHOLD:

        position_fraction = (prob ** SIZE_EXPONENT) * MAX_POSITION

        if position_fraction < MIN_POSITION_FRACTION:
            continue

        invested_amount = paper_capital * INVEST_FRACTION
        qty = invested_amount / current_price

        entry_price = current_price
        entry_time = ts
        position = 1

        trades.append({
            "type": "BUY",
            "time": ts,
            "price": current_price,
            "prob": prob
        })

    # ==============================
    # EXIT
    # ==============================
    elif position == 1:

        pnl_pct = (current_price - entry_price) / entry_price
        hold_minutes = int((ts - entry_time).total_seconds() // 60)

        exit_reason = (
            pnl_pct <= -STOP_LOSS or
            pnl_pct >= TAKE_PROFIT or
            hold_minutes >= HORIZON
        )

        if exit_reason:
            pnl_cash = (current_price - entry_price) * qty
            paper_capital += pnl_cash - invested_amount * COST_PER_TRADE

            trades.append({
                "type": "SELL",
                "time": ts,
                "price": current_price,
                "pnl": pnl_cash
            })

            position = 0
            entry_price = None
            entry_time = None
            qty = 0.0
            invested_amount = 0.0


In [22]:
trades_df = pd.DataFrame(trades)
sell_trades = trades_df[trades_df["type"] == "SELL"]

print("Final Capital:", round(paper_capital, 2))
print("Total Trades:", len(sell_trades))
print("Total PnL:", sell_trades["pnl"].sum())

initial = INITIAL_CAPITAL
final = paper_capital

net_pnl = final - initial
net_return_pct = net_pnl / initial * 100

print(f"Initial Capital : {initial:,.2f}")
print(f"Final Capital   : {final:,.2f}")
print(f"Net PnL         : {net_pnl:,.2f}")
print(f"Net Return (%)  : {net_return_pct:.6f}%")


Final Capital: 1000186.87
Total Trades: 2
Total PnL: 336.87203322446135
Initial Capital : 1,000,000.00
Final Capital   : 1,000,186.87
Net PnL         : 186.87
Net Return (%)  : 0.018687%


In [23]:
# ======================================================
# LIVE PAPER TRADING — 1 MIN INTRADAY (NIFTY BANK)
# CLEAN & BACKTEST-ALIGNED (NO COOLDOWN)
# ======================================================

import yfinance as yf
import time
from datetime import datetime, time as dtime
import pandas as pd
import pytz

# ------------------------------------------------------
# CONFIG
# ------------------------------------------------------
SYMBOL = "^NSEBANK"
INTERVAL = "1m"
IST = pytz.timezone("Asia/Kolkata")

HORIZON = 12
STOP_LOSS = 0.003
TAKE_PROFIT = 0.0045
ENTRY_THRESHOLD = 0.22

INVEST_FRACTION = 0.50
MIN_POSITION_FRACTION = 0.02
SIZE_EXPONENT = 3
MAX_POSITION = 1.0
COST_PER_TRADE = 0.0003

MARKET_OPEN = dtime(9, 15)
MARKET_CLOSE = dtime(15, 15)

# ------------------------------------------------------
# STATE
# ------------------------------------------------------
paper_capital = INITIAL_CAPITAL
position = 0

entry_price = None
entry_time = None
qty = 0.0
invested_amount = 0.0
last_bar_time = None

trades = []

# ------------------------------------------------------
# DATA FETCH
# ------------------------------------------------------
def fetch_1min_data():
    df = yf.download(
        SYMBOL,
        period="2d",
        interval="1m",
        progress=False
    )

    if df.empty:
        return df

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.columns = df.columns.str.capitalize()
    return df.dropna()

# ------------------------------------------------------
# START
# ------------------------------------------------------
print("=" * 80)
print(f"ACTIVE SESSION: {SYMBOL} | LIVE PAPER TRADING")
print("=" * 80)

# ------------------------------------------------------
# LIVE LOOP
# ------------------------------------------------------
while True:
    try:
        now_ist = datetime.now(IST)
        now_time = now_ist.time()

        if now_time < MARKET_OPEN:
            time.sleep(20)
            continue

        if now_time > MARKET_CLOSE:
            print("✅ SESSION ENDED")
            break

        df = fetch_1min_data()

        if df.empty or len(df) < 30:
            time.sleep(10)
            continue

        ts = df.index[-1]
        if ts == last_bar_time:
            time.sleep(5)
            continue

        last_bar_time = ts
        current_price = float(df["Close"].iloc[-1])

        # ------------------------------
        # FEATURE ENGINEERING
        # ------------------------------
        df_feat = add_basic_features(df, horizon=HORIZON, cost=COST_PER_TRADE)

        latest = df_feat.iloc[-1:]
        if latest[feature_cols].isna().any().any():
            time.sleep(5)
            continue

        # ------------------------------
        # ML PREDICTION
        # ------------------------------
        X_live = scaler.transform(latest[feature_cols])
        prob = model.predict_proba(X_live)[0, 1]

        # ------------------------------
        # DASHBOARD
        # ------------------------------
        pnl_pct = 0.0
        state = "SCANNING"

        if position == 1:
            pnl_pct = (current_price - entry_price) / entry_price
            state = "HOLDING"

        print(
            f"\n[{ts.strftime('%H:%M')}] PRICE={current_price:.2f} "
            f"PROB={prob:.3f} STATE={state} "
            f"PnL={pnl_pct*100:+.2f}% CAP={paper_capital:,.0f}"
        )

        # ==============================
        # ENTRY
        # ==============================
        if position == 0 and prob >= ENTRY_THRESHOLD:

            position_fraction = (prob ** SIZE_EXPONENT) * MAX_POSITION

            if position_fraction >= MIN_POSITION_FRACTION:
                invested_amount = paper_capital * INVEST_FRACTION
                qty = invested_amount / current_price

                entry_price = current_price
                entry_time = ts
                position = 1

                trades.append({
                    "type": "BUY",
                    "time": ts,
                    "price": current_price,
                    "prob": prob
                })

                print(f"🚀 BUY @ {current_price:.2f}")

        # ==============================
        # EXIT
        # ==============================
        elif position == 1:

            hold_minutes = int((ts - entry_time).total_seconds() // 60)

            if (
                pnl_pct <= -STOP_LOSS or
                pnl_pct >= TAKE_PROFIT or
                hold_minutes >= HORIZON
            ):
                pnl_cash = (current_price - entry_price) * qty
                paper_capital += pnl_cash - invested_amount * COST_PER_TRADE

                trades.append({
                    "type": "SELL",
                    "time": ts,
                    "price": current_price,
                    "pnl": pnl_cash
                })

                print(f"🏁 SELL @ {current_price:.2f} | PnL ₹{pnl_cash:.2f}")

                position = 0
                entry_price = None
                entry_time = None
                qty = 0.0
                invested_amount = 0.0

        # ------------------------------
        # WAIT NEXT BAR
        # ------------------------------
        time.sleep(max(60 - datetime.now().second, 2))

    except Exception as e:
        print(f"⚠️ RUNTIME ERROR: {e}")
        time.sleep(10)


ACTIVE SESSION: ^NSEBANK | LIVE PAPER TRADING
✅ SESSION ENDED


In [ ]:
# ======================================================
# PAPER TRADING PERFORMANCE ANALYSIS
# ======================================================

import pandas as pd
import numpy as np

paper_df = pd.DataFrame(paper_trades)

if paper_df.empty:
    print("No trades executed.")
else:
    # -------------------------------
    # Separate BUY / SELL trades
    # -------------------------------
    buys = paper_df[paper_df["side"] == "BUY"].reset_index(drop=True)
    sells = paper_df[paper_df["side"].isin(["SELL", "FORCED_SELL"])].reset_index(drop=True)

    trades = pd.concat([buys, sells], axis=1)
    trades = trades.loc[:, ~trades.columns.duplicated()]

    # -------------------------------
    # Returns
    # -------------------------------
    trades["return"] = trades["pnl"]
    trades.dropna(inplace=True)

    total_trades = len(trades)
    wins = trades[trades["return"] > 0]
    losses = trades[trades["return"] <= 0]

    win_rate = len(wins) / total_trades if total_trades > 0 else 0
    avg_win = wins["return"].mean() if not wins.empty else 0
    avg_loss = losses["return"].mean() if not losses.empty else 0

    profit_factor = (
        wins["return"].sum() / abs(losses["return"].sum())
        if not losses.empty else np.inf
    )

    # -------------------------------
    # Equity Curve & Drawdown
    # -------------------------------
    equity = paper_df["capital"].dropna()
    peak = equity.cummax()
    drawdown = (equity - peak) / peak

    max_dd = drawdown.min()

    # -------------------------------
    # Sharpe (intraday approx)
    # -------------------------------
    returns = trades["return"]
    sharpe = (
        np.sqrt(252 * 6.5 * 60) * returns.mean() / returns.std()
        if returns.std() != 0 else 0
    )

    # -------------------------------
    # Print Summary
    # -------------------------------
    print("=" * 60)
    print("INTRADAY PAPER TRADING SUMMARY")
    print("=" * 60)
    print(f"Final Capital      : ₹{paper_capital:,.0f}")
    print(f"Total Trades       : {total_trades}")
    print(f"Win Rate           : {win_rate*100:.2f}%")
    print(f"Avg Win            : {avg_win*100:.3f}%")
    print(f"Avg Loss           : {avg_loss*100:.3f}%")
    print(f"Profit Factor      : {profit_factor:.2f}")
    print(f"Max Drawdown       : {max_dd*100:.2f}%")
    print(f"Sharpe Ratio       : {sharpe:.2f}")
    print("=" * 60)

    trades


In [ ]:
paper_df = pd.DataFrame(paper_trades)
paper_df


In [ ]:
import os
from datetime import datetime

SAVE_DIR = "paper_trades"
os.makedirs(SAVE_DIR, exist_ok=True)

TODAY = datetime.now().strftime("%Y-%m-%d")
CSV_PATH = f"{SAVE_DIR}/niftybank_paper_{TODAY}.csv"

pd.DataFrame(paper_trades).to_csv(CSV_PATH, index=False)
pd.DataFrame(paper_trades).to_csv(CSV_PATH, index=False)

pd.DataFrame(paper_trades).to_csv(CSV_PATH, index=False)
print(f"Trades saved to {CSV_PATH}")
